# Assignment 2

In this assigment, we will work with the *Forest Fire* data set. Please download the data from the [UCI Machine Learning Repository](https://archive.ics.uci.edu/dataset/162/forest+fires). Extract the data files into the subdirectory: `../data/fires/` (relative to `./05_src/`).

## Objective

+ The model objective is to predict the area affected by forest fires given the features set. 
+ The objective of this exercise is to assess your ability to construct and evaluate model pipelines.
+ Please note: the instructions are not meant to be 100% prescriptive, but instead they are a set of minimum requirements. If you find predictive performance gains by applying additional steps, by all means show them. 

## Variable Description

From the description file contained in the archive (`forestfires.names`), we obtain the following variable descriptions:

1. X - x-axis spatial coordinate within the Montesinho park map: 1 to 9
2. Y - y-axis spatial coordinate within the Montesinho park map: 2 to 9
3. month - month of the year: "jan" to "dec" 
4. day - day of the week: "mon" to "sun"
5. FFMC - FFMC index from the FWI system: 18.7 to 96.20
6. DMC - DMC index from the FWI system: 1.1 to 291.3 
7. DC - DC index from the FWI system: 7.9 to 860.6 
8. ISI - ISI index from the FWI system: 0.0 to 56.10
9. temp - temperature in Celsius degrees: 2.2 to 33.30
10. RH - relative humidity in %: 15.0 to 100
11. wind - wind speed in km/h: 0.40 to 9.40 
12. rain - outside rain in mm/m2 : 0.0 to 6.4 
13. area - the burned area of the forest (in ha): 0.00 to 1090.84 









### Specific Tasks

+ Construct four model pipelines, out of combinations of the following components:

    + Preprocessors:

        - A simple processor that only scales numeric variables and recodes categorical variables.
        - A transformation preprocessor that scales numeric variables and applies a non-linear transformation.
    
    + Regressor:

        - A baseline regressor, which could be a [K-nearest neighbours model]() or a linear model like [Lasso](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.Lasso.html) or [Ridge Regressors](https://scikit-learn.org/stable/modules/generated/sklearn.linear_model.ridge_regression.html).
        - An advanced regressor of your choice (e.g., Bagging, Boosting, SVR, etc.). TIP: select a tree-based method such that it does not take too long to run SHAP further below. 

+ Evaluate tune and evaluate each of the four model pipelines. 

    - Select a [performance metric](https://scikit-learn.org/stable/modules/linear_model.html) out of the following options: explained variance, max error, root mean squared error (RMSE), mean absolute error (MAE), r-squared.
    - *TIPS*: 
    
        * Out of the suggested metrics above, [some are correlation metrics, but this is a prediction problem](https://www.tmwr.org/performance#performance). Choose wisely (and don't choose the incorrect options.) 

+ Select the best-performing model and explain its predictions.

    - Provide local explanations.
    - Obtain global explanations and recommend a variable selection strategy.

+ Export your model as a pickle file.


You can work on the Jupyter notebook, as this experiment is fairly short (no need to use sacred). 

# Load the data

Place the files in the ../../05_src/data/fires/ directory and load the appropriate file. 

In [1]:
# Load the libraries as required.
import pandas as pd
import numpy as np

In [2]:
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.linear_model import LinearRegression
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsRegressor
from sklearn.metrics import mean_squared_error

from sklearn.linear_model import Ridge
from sklearn.datasets import make_regression

In [3]:
# Load data
columns = [
    'coord_x', 'coord_y', 'month', 'day', 'ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain', 'area' 
]
fires_dt = (pd.read_csv('../../05_src/data/fires/forestfires.csv', header = 0, names = columns))
fires_dt.info()


<class 'pandas.core.frame.DataFrame'>
RangeIndex: 517 entries, 0 to 516
Data columns (total 13 columns):
 #   Column   Non-Null Count  Dtype  
---  ------   --------------  -----  
 0   coord_x  517 non-null    int64  
 1   coord_y  517 non-null    int64  
 2   month    517 non-null    object 
 3   day      517 non-null    object 
 4   ffmc     517 non-null    float64
 5   dmc      517 non-null    float64
 6   dc       517 non-null    float64
 7   isi      517 non-null    float64
 8   temp     517 non-null    float64
 9   rh       517 non-null    int64  
 10  wind     517 non-null    float64
 11  rain     517 non-null    float64
 12  area     517 non-null    float64
dtypes: float64(8), int64(3), object(2)
memory usage: 52.6+ KB


# Get X and Y

Create the features data frame and target data.

In [4]:
X = fires_dt.drop(columns = 'area')


In [5]:
X. head()

,coord_x,coord_y,month,day,ffmc,dmc,dc,isi,temp,rh,wind,rain
0,7,5,mar,fri,86.2,26.2,94.3,5.1,8.2,51,6.7,0.0
1,7,4,oct,tue,90.6,35.4,669.1,6.7,18.0,33,0.9,0.0
2,7,4,oct,sat,90.6,43.7,686.9,6.7,14.6,33,1.3,0.0
3,8,6,mar,fri,91.7,33.3,77.5,9.0,8.3,97,4.0,0.2
4,8,6,mar,sun,89.3,51.3,102.2,9.6,11.4,99,1.8,0.0


In [6]:
Y = fires_dt['area']

In [7]:
X_train, X_test, Y_train, Y_test = train_test_split(X, Y, test_size = 0.2, random_state = 42)

# Preprocessing

Create two [Column Transformers](https://scikit-learn.org/stable/modules/generated/sklearn.compose.ColumnTransformer.html), called preproc1 and preproc2, with the following guidelines:

- Numerical variables

    * (Preproc 1 and 2) Scaling: use a scaling method of your choice (Standard, Robust, Min-Max). 
    * Preproc 2 only: 
        
        + Choose a transformation for any of your input variables (or several of them). Evaluate if this transformation is convenient.
        + The choice of scaler is up to you.

- Categorical variables: 
    
    * (Preproc 1 and 2) Apply [one-hot encoding](https://scikit-learn.org/stable/modules/generated/sklearn.preprocessing.OneHotEncoder.html) where appropriate.


+ The only difference between preproc1 and preproc2 is the non-linear transformation of the numerical variables.
    


### Preproc 1

Create preproc1 below.

+ Numeric: scaled variables, no other transforms.
+ Categorical: one-hot encoding.

In [8]:
coord_colm = ['coord_x', 'coord_y']
num_colm= ['ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh', 'wind', 'rain']
cat_colm = ['month', 'day']

preproc1 = ColumnTransformer(
    transformers=[
        ('numerical', StandardScaler(), num_colm),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), cat_colm)
    ],
    remainder='passthrough'
)
preproc1

ColumnTransformer(remainder='passthrough',
                  transformers=[('numerical', StandardScaler(),
                                 ['ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh',
                                  'wind', 'rain']),
                                ('categorical',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['month', 'day'])])

### Preproc 2

Create preproc1 below.

+ Numeric: scaled variables, non-linear transformation to one or more variables.
+ Categorical: one-hot encoding.

In [9]:
from sklearn.preprocessing import PowerTransformer

num_colm_transform= ['temp', 'rh', 'wind', 'rain']
pipe_yj = Pipeline([
    ('standardizer', StandardScaler()),
    ('transform', PowerTransformer(method='yeo-johnson'))
])
preproc2 = ColumnTransformer(
    transformers=[
        ('numerical2', pipe_yj, num_colm_transform),
        ('categorical', OneHotEncoder(handle_unknown='ignore'), cat_colm)
    ],
    remainder='passthrough'
)
preproc2

ColumnTransformer(remainder='passthrough',
                  transformers=[('numerical2',
                                 Pipeline(steps=[('standardizer',
                                                  StandardScaler()),
                                                 ('transform',
                                                  PowerTransformer())]),
                                 ['temp', 'rh', 'wind', 'rain']),
                                ('categorical',
                                 OneHotEncoder(handle_unknown='ignore'),
                                 ['month', 'day'])])

## Model Pipeline


Create a [model pipeline](https://scikit-learn.org/stable/modules/generated/sklearn.pipeline.Pipeline.html): 

+ Add a step labelled `preprocessing` and assign the Column Transformer from the previous section.
+ Add a step labelled `regressor` and assign a regression model to it. 

## Regressor

+ Use a regression model to perform a prediction. 

    - Choose a baseline regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Choose a more advance regressor, tune it (if necessary) using grid search, and evaluate it using cross-validation.
    - Both model choices are up to you, feel free to experiment.

In [10]:
model_pipe = Pipeline([
    ('preprocessing', preproc1),
    ('regressor', Ridge())
])
model_pipe

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numerical', StandardScaler(),
                                                  ['ffmc', 'dmc', 'dc', 'isi',
                                                   'temp', 'rh', 'wind',
                                                   'rain']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', Ridge())])

In [11]:
# Pipeline A = preproc1 + baseline


pipeline_A = Pipeline([
    ('preprocessing', preproc1),  # Apply preproc1 
    ('regressor', Ridge())  # Apply Ridge Regression (baseline model)
])
pipeline_A



Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numerical', StandardScaler(),
                                                  ['ffmc', 'dmc', 'dc', 'isi',
                                                   'temp', 'rh', 'wind',
                                                   'rain']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', Ridge())])

In [12]:
# Pipeline B = preproc2 + baseline

pipeline_B = Pipeline([
    ('preprocessing', preproc2),  # Apply preproc2
    ('regressor', Ridge())  # Apply Ridge Regression (baseline model)
])
pipeline_B



Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numerical2',
                                                  Pipeline(steps=[('standardizer',
                                                                   StandardScaler()),
                                                                  ('transform',
                                                                   PowerTransformer())]),
                                                  ['temp', 'rh', 'wind',
                                                   'rain']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', Ridge())])

In [13]:
# Pipeline C = preproc1 + advanced model
from sklearn.ensemble import RandomForestRegressor

pipeline_C = Pipeline([
    ('preprocessing', preproc1),  # Apply preproc1 (scaling & one-hot encoding)
    ('regressor', RandomForestRegressor())  # Apply Random Forest Regression (advanced model)
])
pipeline_C



Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numerical', StandardScaler(),
                                                  ['ffmc', 'dmc', 'dc', 'isi',
                                                   'temp', 'rh', 'wind',
                                                   'rain']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', RandomForestRegressor())])

In [14]:
# Pipeline D = preproc2 + advanced model

pipeline_D = Pipeline([
    ('preprocessing', preproc2),  # Apply preproc2
    ('regressor', RandomForestRegressor())  # Apply Random Forest Regression (advanced model)
])
pipeline_D
    

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numerical2',
                                                  Pipeline(steps=[('standardizer',
                                                                   StandardScaler()),
                                                                  ('transform',
                                                                   PowerTransformer())]),
                                                  ['temp', 'rh', 'wind',
                                                   'rain']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', RandomForestRegressor())])

# Tune Hyperparams

+ Perform GridSearch on each of the four pipelines. 
+ Tune at least one hyperparameter per pipeline.
+ Experiment with at least four value combinations per pipeline.

In [15]:
pipeline_A.get_params()

{'memory': None,
 'steps': [('preprocessing',
   ColumnTransformer(remainder='passthrough',
                     transformers=[('numerical', StandardScaler(),
                                    ['ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh',
                                     'wind', 'rain']),
                                   ('categorical',
                                    OneHotEncoder(handle_unknown='ignore'),
                                    ['month', 'day'])])),
  ('regressor', Ridge())],
 'verbose': False,
 'preprocessing': ColumnTransformer(remainder='passthrough',
                   transformers=[('numerical', StandardScaler(),
                                  ['ffmc', 'dmc', 'dc', 'isi', 'temp', 'rh',
                                   'wind', 'rain']),
                                 ('categorical',
                                  OneHotEncoder(handle_unknown='ignore'),
                                  ['month', 'day'])]),
 'regressor': Ridge(),
 'preprocessing__n_

In [16]:
# pipeline_A

param_grid_A = {
    'regressor__alpha': [0.1, 1.0, 10.0, 100.0],
    'regressor__fit_intercept': [True, False]
}
grid_cv_A = GridSearchCV(
    estimator=pipeline_A, 
    param_grid=param_grid_A, 
    scoring = "neg_root_mean_squared_error", 
    cv = 5)
grid_cv_A.fit(X_train, Y_train)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('numerical',
                                                                         StandardScaler(),
                                                                         ['ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('categorical',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['month',
                                                                          'day'])])),
                                       ('regressor', Ridge())]),
             param_grid={'regressor__alpha': [0.1, 1.0, 10.0, 100.0],
                         'regressor__fit_intercept': [True, False]},
             scoring='neg_root_mean_squared_error')

In [17]:
grid_cv_A.best_params_

{'regressor__alpha': 100.0, 'regressor__fit_intercept': False}

In [18]:
grid_cv_A.best_score_

-39.375286878981896

In [19]:
pd.DataFrame(grid_cv_A.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_regressor__alpha,param_regressor__fit_intercept,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.022006,0.008266,0.009191,0.002315,0.1,True,"{'regressor__alpha': 0.1, 'regressor__fit_inte...",-40.789043,-19.522077,-33.413931,-85.262251,-26.827511,-41.162963,23.146350,8
1,0.013801,0.000749,0.008201,0.000980,0.1,False,"{'regressor__alpha': 0.1, 'regressor__fit_inte...",-40.788502,-19.520043,-33.415314,-85.247861,-26.836563,-41.161657,23.140036,7
2,0.013370,0.001189,0.007202,0.001166,1.0,True,"{'regressor__alpha': 1.0, 'regressor__fit_inte...",-40.846257,-19.269385,-33.565218,-85.243156,-26.535634,-41.091930,23.212720,6
3,0.013596,0.001854,0.007406,0.000800,1.0,False,"{'regressor__alpha': 1.0, 'regressor__fit_inte...",-40.832633,-19.272198,-33.518077,-85.235259,-26.518433,-41.075320,23.214436,5
4,0.012404,0.000484,0.007597,0.001204,10.0,True,"{'regressor__alpha': 10.0, 'regressor__fit_int...",-40.746251,-18.671451,-33.119045,-85.101418,-25.814190,-40.690471,23.391463,4
5,0.012173,0.001333,0.007201,0.001160,10.0,False,"{'regressor__alpha': 10.0, 'regressor__fit_int...",-40.698188,-18.669304,-32.793543,-85.102128,-25.698847,-40.592402,23.428143,3
6,0.013998,0.001095,0.008398,0.000799,100.0,True,"{'regressor__alpha': 100.0, 'regressor__fit_in...",-40.166910,-17.525175,-30.573586,-84.758995,-24.476283,-39.500190,23.820610,2
7,0.012398,0.001021,0.007401,0.000488,100.0,False,"{'regressor__alpha': 100.0, 'regressor__fit_in...",-40.112548,-17.525243,-30.133278,-84.752112,-24.353253,-39.375287,23.866715,1


In [20]:
# pipeline_B

param_grid_B = {
    'regressor__alpha': [0.1, 1.0, 10.0, 100.0],
    'regressor__fit_intercept': [True, False]
}
grid_cv_B = GridSearchCV(
    estimator=pipeline_B, 
    param_grid=param_grid_B, 
    scoring = "neg_root_mean_squared_error", 
    cv = 5)
grid_cv_B.fit(X_train, Y_train)



GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('numerical2',
                                                                         Pipeline(steps=[('standardizer',
                                                                                          StandardScaler()),
                                                                                         ('transform',
                                                                                          PowerTransformer())]),
                                                                         ['temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('categorical',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['month',
                                                                          'day'])])),
                                       ('regressor', Ridge())]),
             param_grid={'regressor__alpha': [0.1, 1.0, 10.0, 100.0],
                         'regressor__fit_intercept': [True, False]},
             scoring='neg_root_mean_squared_error')

In [21]:
grid_cv_B.best_params_

{'regressor__alpha': 100.0, 'regressor__fit_intercept': False}

In [22]:
grid_cv_B.best_score_

-39.003218678138936

In [23]:
pd.DataFrame(grid_cv_B.cv_results_)

,mean_fit_time,std_fit_time,mean_score_time,std_score_time,param_regressor__alpha,param_regressor__fit_intercept,params,split0_test_score,split1_test_score,split2_test_score,split3_test_score,split4_test_score,mean_test_score,std_test_score,rank_test_score
0,0.028801,0.007194,0.013199,0.010046,0.1,True,"{'regressor__alpha': 0.1, 'regressor__fit_inte...",-40.913091,-19.541464,-29.124234,-85.217748,-26.803955,-40.320098,23.477375,8
1,0.022402,0.001361,0.007201,0.002037,0.1,False,"{'regressor__alpha': 0.1, 'regressor__fit_inte...",-40.924518,-19.401390,-29.121146,-85.210207,-26.689111,-40.269274,23.512920,7
2,0.027400,0.001619,0.008601,0.001207,1.0,True,"{'regressor__alpha': 1.0, 'regressor__fit_inte...",-40.944569,-19.328905,-29.030160,-85.200663,-26.499686,-40.200797,23.552837,6
3,0.027991,0.001430,0.008624,0.000462,1.0,False,"{'regressor__alpha': 1.0, 'regressor__fit_inte...",-40.953644,-19.117691,-29.034337,-85.165704,-26.395115,-40.133298,23.588857,5
4,0.026202,0.002135,0.008399,0.001356,10.0,True,"{'regressor__alpha': 10.0, 'regressor__fit_int...",-40.797513,-18.796885,-28.683831,-85.069453,-25.778048,-39.825146,23.713416,4
5,0.024405,0.002329,0.008600,0.001495,10.0,False,"{'regressor__alpha': 10.0, 'regressor__fit_int...",-40.780957,-18.675889,-28.687651,-85.017097,-25.725018,-39.777322,23.720721,3
6,0.023633,0.001828,0.006966,0.001292,100.0,True,"{'regressor__alpha': 100.0, 'regressor__fit_in...",-40.151298,-17.693095,-28.004374,-84.690771,-24.542472,-39.016402,23.970310,2
7,0.024398,0.002653,0.008801,0.000745,100.0,False,"{'regressor__alpha': 100.0, 'regressor__fit_in...",-40.134194,-17.675464,-27.999750,-84.665194,-24.541491,-39.003219,23.964082,1


In [24]:
# pipeline_C

param_grid_C = {
    'regressor__n_estimators': [50, 100, 150, 200],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__bootstrap': [True, False]
}
grid_cv_C = GridSearchCV(
    estimator=pipeline_C, 
    param_grid=param_grid_C, 
    cv = 5,
    scoring='neg_root_mean_squared_error'
)
grid_cv_C.fit(X_train, Y_train)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('numerical',
                                                                         StandardScaler(),
                                                                         ['ffmc',
                                                                          'dmc',
                                                                          'dc',
                                                                          'isi',
                                                                          'temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('categorical',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['month',
                                                                          'day'])])),
                                       ('regressor', RandomForestRegressor())]),
             param_grid={'regressor__bootstrap': [True, False],
                         'regressor__max_depth': [None, 10, 20, 30],
                         'regressor__n_estimators': [50, 100, 150, 200]},
             scoring='neg_root_mean_squared_error')

In [25]:
grid_cv_C.best_params_

{'regressor__bootstrap': True,
 'regressor__max_depth': 10,
 'regressor__n_estimators': 100}

In [26]:
grid_cv_C.best_score_

-47.55618110198704

In [27]:
# pipeline_D

param_grid_D = {
    'regressor__n_estimators': [50, 100, 150, 200],
    'regressor__max_depth': [None, 10, 20, 30],
    'regressor__bootstrap': [True, False]
}
grid_cv_D = GridSearchCV(
    estimator=pipeline_D, 
    param_grid=param_grid_D, 
    cv = 5,
    scoring="neg_root_mean_squared_error"
)
grid_cv_D.fit(X_train, Y_train)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('preprocessing',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('numerical2',
                                                                         Pipeline(steps=[('standardizer',
                                                                                          StandardScaler()),
                                                                                         ('transform',
                                                                                          PowerTransformer())]),
                                                                         ['temp',
                                                                          'rh',
                                                                          'wind',
                                                                          'rain']),
                                                                        ('categorical',
                                                                         OneHotEncoder(handle_unknown='ignore'),
                                                                         ['month',
                                                                          'day'])])),
                                       ('regressor', RandomForestRegressor())]),
             param_grid={'regressor__bootstrap': [True, False],
                         'regressor__max_depth': [None, 10, 20, 30],
                         'regressor__n_estimators': [50, 100, 150, 200]},
             scoring='neg_root_mean_squared_error')

In [29]:
grid_cv_D.best_params_

{'regressor__bootstrap': True,
 'regressor__max_depth': 20,
 'regressor__n_estimators': 200}

In [30]:
grid_cv_D.best_score_

-47.4472505243718

# Evaluate

+ Which model has the best performance?

Based on the RMSE scores of the 4 models, the second one, grid_cv_B with alpha = 100 and fit_intercept = False has the best performance.

In [31]:
grid_cv_B.best_estimator_

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numerical2',
                                                  Pipeline(steps=[('standardizer',
                                                                   StandardScaler()),
                                                                  ('transform',
                                                                   PowerTransformer())]),
                                                  ['temp', 'rh', 'wind',
                                                   'rain']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', Ridge(alpha=100.0, fit_intercept=False))])

# Export

+ Save the best performing model to a pickle file.

In [36]:
import os
import pickle

In [37]:

fire_dir = os.getenv('FIRE_DIR','./')
os.makedirs(fire_dir, exist_ok=True)
    
output = os.path.join(
    fire_dir, 
    "best_model.pkl")
    
with open(output, 'wb') as f:
    pickle.dump(grid_cv_B.best_estimator_, f)

# Explain

+ Use SHAP values to explain the following only for the best-performing model:

    - Select an observation in your test set and explain which are the most important features that explain that observation's specific prediction.

    - In general, across the complete training set, which features are the most and least important.

+ If you were to remove features from the model, which ones would you remove? Why? How would you test that these features are actually enhancing model performance?

In [38]:
import shap

pickle_file = 'best_model.pkl'

with open(pickle_file, 'rb') as file:
    model = pickle.load(file)

model

Pipeline(steps=[('preprocessing',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('numerical2',
                                                  Pipeline(steps=[('standardizer',
                                                                   StandardScaler()),
                                                                  ('transform',
                                                                   PowerTransformer())]),
                                                  ['temp', 'rh', 'wind',
                                                   'rain']),
                                                 ('categorical',
                                                  OneHotEncoder(handle_unknown='ignore'),
                                                  ['month', 'day'])])),
                ('regressor', Ridge(alpha=100.0, fit_intercept=False))])

*(Answer here.)*

## Criteria

The [rubric](./assignment_2_rubric_clean.xlsx) contains the criteria for assessment.

## Submission Information

🚨 **Please review our [Assignment Submission Guide](https://github.com/UofT-DSI/onboarding/blob/main/onboarding_documents/submissions.md)** 🚨 for detailed instructions on how to format, branch, and submit your work. Following these guidelines is crucial for your submissions to be evaluated correctly.

### Submission Parameters:
* Submission Due Date: `HH:MM AM/PM - DD/MM/YYYY`
* The branch name for your repo should be: `assignment-2`
* What to submit for this assignment:
    * This Jupyter Notebook (assignment_2.ipynb) should be populated and should be the only change in your pull request.
* What the pull request link should look like for this assignment: `https://github.com/<your_github_username>/production/pull/<pr_id>`
    * Open a private window in your browser. Copy and paste the link to your pull request into the address bar. Make sure you can see your pull request properly. This helps the technical facilitator and learning support staff review your submission easily.

Checklist:
- [ ] Created a branch with the correct naming convention.
- [ ] Ensured that the repository is public.
- [ ] Reviewed the PR description guidelines and adhered to them.
- [ ] Verify that the link is accessible in a private browser window.

If you encounter any difficulties or have questions, please don't hesitate to reach out to our team via our Slack at the `help` channel. Our Technical Facilitators and Learning Support staff are here to help you navigate any challenges.

# Reference

Cortez,Paulo and Morais,Anbal. (2008). Forest Fires. UCI Machine Learning Repository. https://doi.org/10.24432/C5D88D.